# tensor-item-scalar — worked example 1: .item() — extract loss scalar for logging list

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `tensor-item-scalar`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

In a training loop, each forward pass produces a loss as a 0-D tensor. Logging it directly by appending the tensor to a list accumulates the computation graph, leaking memory over thousands of steps. Calling `.item()` extracts a plain Python float that holds only the value, breaking the graph link and keeping the log list lightweight.

## Worked solution

**Step 1 — Why not just append the tensor?**
A 0-D tensor still holds a reference to its `grad_fn`, which in turn holds the entire computation graph for that step. Appending N tensors to a list keeps N graphs alive — memory grows with every step.

**Step 2 — `.item()` extracts the value.**
`loss.item()` returns a Python `float`. It has no `grad_fn` and no memory beyond the float itself. Appending it to a list is effectively free.

**Step 3 — Use `.item()` only outside the forward/backward path.**
Do not call `.item()` on tensors you still need for `.backward()`. Call it after `backward()`, or on the loss copy before zeroing gradients.

**Step 4 — Implementation.**
Run the forward pass, call `.item()` to capture the scalar, then proceed with `.backward()`. The log list accumulates floats.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(42)

# Toy model: single linear layer
model = nn.Linear(4, 1)
optimizer = t.optim.SGD(model.parameters(), lr=0.01)

# Fake data
t.manual_seed(42)
X = t.randn(20, 4)
y = t.randn(20, 1)

loss_log = []
for step in range(10):
    optimizer.zero_grad()
    pred = model(X)
    loss = nn.functional.mse_loss(pred, y)
    loss_log.append(loss.item())  # <-- extract float BEFORE backward
    loss.backward()
    optimizer.step()

print('Loss log type:', type(loss_log[0]))   # <class 'float'>
print('Loss at step 0:', loss_log[0])
print('Loss at step 9:', loss_log[9])
print('Loss decreased:', loss_log[0] > loss_log[-1])